In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize


# =========================================================
# 1. Load data
# =========================================================
mutualfunds_path = "mutualfunds.csv"
mktetf_path = "mktetf.csv"
output_path = "q4_style_benchmark_output.xlsx"

fund_df = pd.read_csv(mutualfunds_path)
etf_df = pd.read_csv(mktetf_path)

# Standardize date columns
fund_df.columns = [c.strip() for c in fund_df.columns]
etf_df.columns = [c.strip() for c in etf_df.columns]

# Try to detect date column names
fund_date_col = "date" if "date" in [c.lower() for c in fund_df.columns] else fund_df.columns[0]
etf_date_col = "Date" if "Date" in etf_df.columns else etf_df.columns[0]

if fund_date_col != "date":
    fund_df = fund_df.rename(columns={fund_date_col: "date"})
if etf_date_col != "date":
    etf_df = etf_df.rename(columns={etf_date_col: "date"})

fund_df["date"] = pd.to_datetime(fund_df["date"])
etf_df["date"] = pd.to_datetime(etf_df["date"])

# Merge
df = pd.merge(fund_df, etf_df, on="date", how="inner").sort_values("date").reset_index(drop=True)

In [2]:
df.head()

,date,LCGFX,MRLIX,mktvw,rf,EEM,EFA,IWD,IWF,IWN,IWO,LQD,SHY,TLT
0,2016-01-29,-0.061437,-0.068841,-0.057090,0.0001,-0.050326,-0.055177,-0.052831,-0.056795,-0.065477,-0.105974,0.001228,0.006520,0.055731
1,2016-02-29,-0.015106,0.009728,0.000620,0.0002,-0.008178,-0.033345,0.000000,0.000000,0.006052,-0.008432,0.010528,0.001121,0.030866
2,2016-03-31,0.066462,0.067437,0.070506,0.0002,0.129617,0.065821,0.072667,0.067362,0.082707,0.076195,0.036162,0.001376,-0.000896
3,2016-04-29,-0.003835,-0.016245,0.011719,0.0001,0.004088,0.022218,0.021051,-0.009220,0.021357,0.011010,0.015539,0.000361,-0.007368
4,2016-05-31,0.028874,0.018349,0.014341,0.0001,-0.036929,-0.000856,0.014669,0.019624,0.017863,0.027672,-0.005124,-0.001188,0.008077


In [3]:
# =========================================================
# 2. Define ETF columns used in style analysis
# =========================================================
# Based on assignment / style analysis template
etf_cols = ["IWF", "IWD", "IWO", "IWN", "TLT", "SHY", "LQD", "EFA", "EEM"]

# Check columns exist
missing_cols = [c for c in etf_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing ETF columns in merged data: {missing_cols}")

fund_cols = ["LCGFX", "MRLIX"]
missing_funds = [c for c in fund_cols if c not in df.columns]
if missing_funds:
    raise ValueError(f"Missing fund columns in merged data: {missing_funds}")

In [4]:
# =========================================================
# 3. Optimization function
# =========================================================
def estimate_style_weights(y, X):
    """
    Minimize residual variance (equivalently SSE for fixed sample size)
    subject to:
      - weights >= 0
      - weights <= 1
      - sum(weights) = 1
    """
    n_assets = X.shape[1]

    def objective(w):
        residuals = y - X @ w
        return np.var(residuals, ddof=0)

    x0 = np.full(n_assets, 1.0 / n_assets)

    bounds = [(0.0, 1.0) for _ in range(n_assets)]
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    result = minimize(
        objective,
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints
    )

    if not result.success:
        raise RuntimeError(f"Optimization failed: {result.message}")

    return result.x

In [5]:
# =========================================================
# 4. Rolling benchmark construction
# =========================================================
def build_style_benchmark(data, fund_col, etf_cols, start_date="2021-01-01", end_date="2025-12-31", window=36):
    """
    For each month t in [start_date, end_date]:
      - use previous 36 months to estimate style weights
      - use month t ETF returns and estimated weights to compute benchmark return
      - save exposures, benchmark return, and fund return
    """
    data = data.sort_values("date").reset_index(drop=True)

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    results = []

    for i in range(window, len(data)):
        current_date = data.loc[i, "date"]

        if current_date < start_date or current_date > end_date:
            continue

        # previous 36 months
        train = data.iloc[i - window:i].copy()

        y = train[fund_col].to_numpy(dtype=float)
        X = train[etf_cols].to_numpy(dtype=float)

        # estimate weights
        weights = estimate_style_weights(y, X)

        # month t benchmark return
        current_etf_returns = data.loc[i, etf_cols].to_numpy(dtype=float)
        benchmark_return = float(np.dot(weights, current_etf_returns))
        fund_return = float(data.loc[i, fund_col])

        row = {
            "date": current_date,
            "fund": fund_col,
            "benchmark_return": benchmark_return,
            "fund_return": fund_return,
            "active_return": fund_return - benchmark_return
        }

        for col, w in zip(etf_cols, weights):
            row[col] = w

        results.append(row)

    return pd.DataFrame(results)

In [6]:
# =========================================================
# 5. Run for both funds
# =========================================================
lcgfx_out = build_style_benchmark(df, "LCGFX", etf_cols)
mrlix_out = build_style_benchmark(df, "MRLIX", etf_cols)

In [10]:
# =========================================================
# 6. Save output
# =========================================================
# Excel output (combined)
with pd.ExcelWriter("q4_style_benchmark_output.xlsx", engine="openpyxl") as writer:
    lcgfx_out.to_excel(writer, sheet_name="LCGFX_Q4a", index=False)
    mrlix_out.to_excel(writer, sheet_name="MRLIX_Q4a", index=False)

# CSV outputs (separate)
lcgfx_out.to_csv("q4a_LCGFX.csv", index=False)
mrlix_out.to_csv("q4a_MRLIX.csv", index=False)

print("Done.")
print(f"Saved output to: {output_path}")

Done.
Saved output to: q4_style_benchmark_output.xlsx


In [11]:
# =========================================================
# 7. Optional: print Q4(b) summary stats
# =========================================================
def summarize_ir(df_out):
    avg_active_return = df_out["active_return"].mean()
    tracking_error = df_out["active_return"].std(ddof=1)
    information_ratio = avg_active_return / tracking_error if tracking_error != 0 else np.nan
    return avg_active_return, tracking_error, information_ratio

for name, out in [("LCGFX", lcgfx_out), ("MRLIX", mrlix_out)]:
    avg_ar, te, ir = summarize_ir(out)
    print(f"\n{name}")
    print(f"Average active return: {avg_ar:.6f}")
    print(f"Tracking error:      {te:.6f}")
    print(f"Information ratio:   {ir:.6f}")


LCGFX
Average active return: -0.000950
Tracking error:      0.016850
Information ratio:   -0.056364

MRLIX
Average active return: 0.002253
Tracking error:      0.017168
Information ratio:   0.131208


In [14]:
benchmark_df = pd.read_csv("q4a_LCGFX.csv")
benchmark_df

,date,fund,benchmark_return,fund_return,active_return,IWF,IWD,IWO,IWN,TLT,SHY,LQD,EFA,EEM
0,2021-01-29,LCGFX,0.001809,-0.031470,-0.033279,0.442199,1.489472e-01,0.209863,1.558952e-17,6.784798e-02,3.557308e-02,1.135848e-02,8.421103e-02,6.938894e-18
1,2021-02-26,LCGFX,0.012005,0.043162,0.031157,0.439059,1.662623e-01,0.192623,4.364314e-17,8.655439e-02,4.368129e-03,3.046032e-02,8.067284e-02,0.000000e+00
2,2021-03-31,LCGFX,0.010244,0.011158,0.000914,0.444390,1.797690e-01,0.178188,7.847480e-18,9.519510e-02,1.734723e-18,2.299236e-02,7.946569e-02,0.000000e+00
3,2021-04-30,LCGFX,0.045065,0.068506,0.023441,0.429463,1.647447e-01,0.192896,1.165950e-17,1.113611e-01,2.602085e-18,2.647731e-02,7.505761e-02,3.035766e-18
4,2021-05-28,LCGFX,-0.003989,-0.001721,0.002268,0.423694,1.679995e-01,0.177659,0.000000e+00,1.201255e-01,0.000000e+00,3.068443e-02,7.983778e-02,0.000000e+00
5,2021-06-30,LCGFX,0.037360,0.049138,0.011778,0.422865,1.671749e-01,0.190778,0.000000e+00,1.114045e-01,8.673617e-18,3.057241e-02,7.720482e-02,1.098658e-17
6,2021-07-30,LCGFX,0.013578,0.036976,0.023398,0.422204,1.547157e-01,0.199118,2.953218e-19,1.174747e-01,0.000000e+00,3.582114e-02,7.066628e-02,0.000000e+00
7,2021-08-31,LCGFX,0.024649,0.027338,0.002689,0.465978,1.753268e-01,0.186850,3.963244e-17,8.455349e-02,2.086313e-03,9.904901e-03,7.530048e-02,0.000000e+00
8,2021-09-30,LCGFX,-0.044568,-0.051292,-0.006724,0.459760,1.775558e-01,0.179348,0.000000e+00,8.209600e-02,1.015872e-03,1.283917e-02,8.738541e-02,0.000000e+00
9,2021-10-29,LCGFX,0.062473,0.090650,0.028177,0.455159,1.766586e-01,0.182688,3.491820e-17,8.583646e-02,3.450443e-18,1.038318e-02,8.927481e-02,2.096124e-18
